---
title: Matching Bedmap points to OPR frames
date: 2026-04-28
---

Bedmap is a community-maintained point cloud of bed-elevation picks compiled from many surveys; xOPR exposes radar-derived bed picks frame-by-frame. To compare them or fuse them, we need a fast spatial join between two heterogeneous datasets:

- **Bedmap** — millions of `(lat, lon, bed_elev)` points, no native frame association.
- **OPR** — flight-line GeoDataFrame whose every STAC item carries an `opr:mbox` (4 morton cells covering the line geometry).

This notebook walks through the two-stage matching API:

1. **Coarse match** (`xopr.bedmap.match_bedmap_to_frames`) — for each Bedmap point, encode an order-18 morton index and find every OPR frame whose `opr:mbox` contains that point via prefix-string containment. Same `bedmap_str.startswith(mbox_str)` invariant the morton extension is built around.
2. **Disambiguate** (`xopr.bedmap.disambiguate_matches`) — when a point matches multiple frames (typical at frame boundaries, or where flight lines self-intersect), use along-track runs to assign each point to a specific frame and compute the nearest OPR layer pick.

We also use [`xopr.bedmap.classify_ambiguity`](../api/xopr.bedmap.html#xopr.bedmap.classify_ambiguity) to diagnose whether the multi-match cases are benign frame-boundary overlaps or genuine cross-segment overlaps from revisits.

**Companion notebooks**: see [`bed_picks.ipynb`](./bed_picks.ipynb) for loading OPR bed picks generally, [`bedmachine_comparison.ipynb`](./bedmachine_comparison.ipynb) for OPR vs raster comparison, and [`xopr_atl06_crossovers.ipynb`](./xopr_atl06_crossovers.ipynb) for the polygon-vs-frame variant of this same prefix trick.

## 1. Load the catalog and Bedmap data

In [ ]:
import time

import fsspec
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

import xopr
from xopr.bedmap import (
    classify_ambiguity,
    disambiguate_matches,
    fetch_bedmap,
    match_bedmap_to_frames,
    query_bedmap,
)
from xopr.stac.morton import mbox_to_polygons

In [ ]:
# Post-#78 catalog with opr:mbox per item. Pick a season covered by Bedmap.
CATALOG_URL = (
    "https://data.source.coop/englacial/xopr/catalog/"
    "hemisphere=south/provider=cresis/collection=2009_Antarctica_DC8/stac.parquet"
)

with fsspec.open(CATALOG_URL) as f:
    stac_gdf = gpd.read_parquet(f)

if 'opr:mbox' not in stac_gdf.columns:
    raise RuntimeError(
        'Catalog has no opr:mbox column — it predates PR #77 / issue #78. '
        'Pick a reprocessed season from https://github.com/englacial/xopr/issues/78.'
    )

print(f"Loaded {len(stac_gdf)} frames from {stac_gdf['collection'].iloc[0]}")
stac_gdf[['id', 'opr:date', 'opr:segment', 'opr:frame', 'opr:mbox']].head(3)

In [ ]:
# Fetch matching Bedmap points within the catalog footprint.
# query_bedmap streams from source.coop and supports spatial pre-filtering.
footprint = stac_gdf.unary_union
fetch_bedmap('bedmap2')
bedmap_gdf = query_bedmap(
    geometry=footprint,
    collections=['bedmap2'],
    local_cache=True,
)
# query_bedmap returns a DataFrame with lat/lon columns; build a GeoDataFrame.
bedmap_gdf = gpd.GeoDataFrame(
    bedmap_gdf,
    geometry=gpd.points_from_xy(bedmap_gdf['lon'], bedmap_gdf['lat']),
    crs='EPSG:4326',
)
print(f"Bedmap points in footprint: {len(bedmap_gdf):,}")

## 2. Morton-prefix matching (the coarse stage)

[`match_bedmap_to_frames`](../api/xopr.bedmap.html#xopr.bedmap.match_bedmap_to_frames) computes an order-18 morton index for every Bedmap point, then flags every OPR frame whose mbox cell is a prefix of that index. Variable-resolution cells: a 5-digit cell matches a wide swath of bedmap points; a 19-digit cell matches only points within ~5m of the frame line.

In [ ]:
t0 = time.time()
result = match_bedmap_to_frames(stac_gdf, bedmap_gdf)
print(f"Matching: {time.time() - t0:.1f}s for {len(bedmap_gdf):,} points × {len(stac_gdf)} frames")

total = len(result)
matched = (result['n_candidates'] > 0).sum()
unique = (result['n_candidates'] == 1).sum()
ambig = (result['n_candidates'] > 1).sum()
unmatched = (result['n_candidates'] == 0).sum()

print(f"\n  Matched (≥1 candidate): {matched:>8,} ({matched/total:.1%})")
print(f"  Unambiguous (1):       {unique:>8,} ({unique/total:.1%})")
print(f"  Ambiguous (>1):        {ambig:>8,} ({ambig/total:.1%})")
print(f"  Unmatched (0):         {unmatched:>8,} ({unmatched/total:.1%})")

### Why are points ambiguous?

[`classify_ambiguity`](../api/xopr.bedmap.html#xopr.bedmap.classify_ambiguity) buckets each multi-match into one of:

- `unique` — exactly one candidate (no ambiguity).
- `consecutive` — adjacent frames within a single segment (a benign frame-boundary overlap).
- `cross_segment` — multiple segments, multiple dates, or non-adjacent frames within one segment (overlapping flight lines or revisits).

In [ ]:
result['ambiguity'] = result['opr_candidates'].apply(classify_ambiguity)
print(result['ambiguity'].value_counts())

## 3. Visualize the mbox coverage and match outcome

In [ ]:
# Materialize every frame's 4 mbox cells as shapely Polygons in EPSG:4326.
mbox_records = []
for _, row in stac_gdf.iterrows():
    for poly in mbox_to_polygons(row['opr:mbox']):
        mbox_records.append({
            'geometry': poly,
            'opr:frame': row['opr:frame'],
            'opr:date': row['opr:date'],
        })
mbox_gdf = gpd.GeoDataFrame(mbox_records, crs='EPSG:4326').to_crs('EPSG:3031')
stac_proj = stac_gdf.to_crs('EPSG:3031')

# Subsample bedmap for plotting speed.
sample_n = min(50_000, len(result))
plot_sample = result.sample(sample_n, random_state=42).to_crs('EPSG:3031')
groups = {
    'unmatched': (plot_sample[plot_sample['n_candidates'] == 0], '#e74c3c'),
    'unique':    (plot_sample[plot_sample['n_candidates'] == 1], '#2ecc71'),
    'ambiguous': (plot_sample[plot_sample['n_candidates'] > 1],  '#f39c12'),
}

fig, ax = plt.subplots(figsize=(12, 12))
mbox_gdf.plot(ax=ax, facecolor='cornflowerblue', edgecolor='navy',
              linewidth=0.2, alpha=0.15, label='mbox cells')
for label, (gdf, color) in groups.items():
    gdf.plot(ax=ax, color=color, markersize=0.5, alpha=0.4,
             label=f'{label} ({len(gdf):,})')
stac_proj.plot(ax=ax, color='blue', linewidth=0.3, alpha=0.5,
               label=f'OPR frames ({len(stac_gdf)})')
ax.legend(loc='upper right', markerscale=15, fontsize=9)
ax.set_title('Morton mbox coverage and match results (EPSG:3031)')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
plt.tight_layout()

## 4. Disambiguate: assign each point to a single frame

[`disambiguate_matches`](../api/xopr.bedmap.html#xopr.bedmap.disambiguate_matches) takes the multi-candidate output, groups adjacent frames, and assigns each point to a specific frame using along-track contiguous runs. Within each assigned group, it then finds the nearest layer pick to compute distance.

We need a layer-pick GeoDataFrame to feed in. [`OPRConnection.load_bed_picks`](../api/xopr.opr_access.html#xopr.opr_access.OPRConnection.load_bed_picks) produces exactly that schema — see the [bed_picks notebook](./bed_picks.ipynb) for details.

In [ ]:
opr = xopr.OPRConnection(cache_dir='radar_cache')
# Load bed picks for every frame in the catalog. This takes a while (one network call per frame).
layer_gdf = opr.load_bed_picks(stac_gdf, target_crs='EPSG:3031')
print(f"Loaded {len(layer_gdf):,} OPR layer picks")

In [ ]:
t0 = time.time()
disambig = disambiguate_matches(result, layer_gdf, stac_gdf, group_size=5)
print(f"Disambiguation: {time.time() - t0:.1f}s")

d = disambig['nearest_distance_m']
print(f"\nDistance from each Bedmap point to its assigned OPR layer pick:")
print(f"  median: {d.median():>7.0f} m")
print(f"  p80:    {d.quantile(0.80):>7.0f} m")
print(f"  p95:    {d.quantile(0.95):>7.0f} m")

## 5. Group-size sensitivity

`group_size` controls how many adjacent frames are bundled together when forming along-track runs. Larger groups give a wider spatial filter (fewer, longer runs) but coarser frame assignment; smaller groups are finer but produce more, shorter runs.

In [ ]:
rows = []
for gs in (3, 5, 7, 10):
    t0 = time.time()
    out = disambiguate_matches(result, layer_gdf, stac_gdf, group_size=gs)
    elapsed = time.time() - t0
    d = out['nearest_distance_m']
    rl = out['run_length']
    rl_pos = rl[rl > 0]
    rows.append({
        'group_size': gs,
        'n_runs': int(out['run_id'].max() + 1),
        'median_run_len': int(rl_pos.median()),
        'p90_run_len': int(rl_pos.quantile(0.90)),
        'median_dist_m': int(d.median()),
        'p95_dist_m': int(d.quantile(0.95)),
        'time_s': round(elapsed, 1),
    })
import pandas as pd
pd.DataFrame(rows).set_index('group_size')

## 6. Summary

- **Coarse match** (`match_bedmap_to_frames`) is fast: ~$O(\text{points} \times \text{distinct prefix lengths})$ via a string-prefix lookup, no polygon intersection.
- **Most ambiguity is benign**: `classify_ambiguity` shows the bulk of multi-match cases are `consecutive` frame-boundary overlaps. The interesting cases are `cross_segment` revisits.
- **Disambiguation** picks a single frame per point using contiguous along-track runs, then nearest-neighbor to the actual layer picks. With `group_size=5`, ~94% of points are within 100m of their assigned OPR layer pick at this scale.
- The output schema `(assigned_segment_path, assigned_frame, nearest_distance_m, run_id, run_length)` is the basis for further analysis: filter by `nearest_distance_m` for high-confidence pairings, or by `run_length` for points belonging to long along-track segments.